In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [4]:
def exp_func(lmbd, p):
    return -1 / lmbd * np.log(p)

def generate_request_time_exp(lmbd, work_time):
    T_server = [0]
    while True:
        new_t = T_server[-1] + exp_func(lmbd, np.random.rand())
        if new_t > work_time:
            break
        T_server.append(new_t)
    T_server.pop(0)
    return np.asarray(T_server)

def generate_process_time_exp(lmbd, n):
    return exp_func(lmbd, np.random.rand(n))

def get_f_o_t(cur_t, last_start_s1, last_end_s1, last_start_s2, last_end_s2):
    last_t = max(last_start_s1, last_start_s2)
    two = max(0, (min(last_end_s1, last_end_s2) - last_t))
    free = max(0, cur_t - max(last_end_s1, last_end_s2))
    one = cur_t - last_t - two - free
    return (free, one, two)

def simulate(Ts, Tz, work_time):
    last_start_s1 = 0
    last_end_s1 = 0
    last_start_s2 = 0
    last_end_s2 = 0
    processed_signals = 0

    free_time = 0
    one_work_time = 0
    two_work_time = 0

    first_ch_time = 0
    first_ch_c = 0
    second_ch_time = 0
    second_ch_c = 0

    for i in range(len(Ts)):
        if Ts[i] + Tz[i] <= work_time:
            if last_end_s1 <= Ts[i]:
                free, one, two = get_f_o_t(Ts[i], last_start_s1, last_end_s1, last_start_s2, last_end_s2)
                free_time += free
                one_work_time += one
                two_work_time += two

                last_start_s1, last_end_s1 = Ts[i], Ts[i] + Tz[i]
                processed_signals += 1
                first_ch_time += Tz[i]
                first_ch_c += 1
            elif last_end_s2 <= Ts[i]:
                free, one, two = get_f_o_t(Ts[i], last_start_s1, last_end_s1, last_start_s2, last_end_s2)
                free_time += free
                one_work_time += one
                two_work_time += two

                last_start_s2, last_end_s2 = Ts[i], Ts[i] + Tz[i]
                processed_signals += 1
                second_ch_time += Tz[i]
                second_ch_c += 1
    
    free, one, two = get_f_o_t(work_time, last_start_s1, last_end_s1, last_start_s2, last_end_s2)
    free_time += free
    one_work_time += one
    two_work_time += two

    return (processed_signals, (free_time, one_work_time, two_work_time), (first_ch_time, first_ch_c, second_ch_time, second_ch_c))

In [5]:
lmbd = 0.2
mu = 0.1
T = 1000

In [6]:
T_req = generate_request_time_exp(lmbd, T)
T_processing = generate_process_time_exp(mu, len(T_req))

processed_requests, fot_times, ffss = simulate(T_req, T_processing, T)

print("============Симуляция============")
print(f"Число обработанных программ: {processed_requests}")
print(f"Число отказов: {len(T_req) - processed_requests}")
print(f"Веросятность обработки: {processed_requests / len(T_req)}")
print(f"Веросятность отказа: {1 - (processed_requests / len(T_req))}")
print(f"Время простоя первого канала: {np.sum(fot_times) - ffss[0]}")
print(f"Время простоя второго канала: {np.sum(fot_times) - ffss[2]}")
print(f"Вероятность загрузки первого канала: {ffss[0] / np.sum(fot_times)}")
print(f"Вероятность загрузки второго канала: {ffss[2] / np.sum(fot_times)}")
print(f"Вероятность загрузки первого или второго канала: {1 - (fot_times[0] / np.sum(fot_times))}")

============Симуляция============
Число обработанных программ: 127
Число отказов: 73
Веросятность обработки: 0.635
Веросятность отказа: 0.365
Время простоя первого канала: 378.08685748527193
Время простоя второго канала: 449.7667243175563
Вероятность загрузки первого канала: 0.6219131425147281
Вероятность загрузки второго канала: 0.5502332756824437
Вероятность загрузки первого или второго канала: 0.8036817471589155


In [20]:
test_p = lmbd / mu

p0 = 1 / (1 + test_p + test_p**2 / 2)
p1 = test_p * p0
p2 = test_p**2 / 2 * p0

Q = 1 - p2
A = lmbd * Q
k = A / mu

print(p0)
print(p1)
print(p2)
print('========')
print(Q)
print(A)
print(k)

0.2
0.4
0.4
0.6
0.12
1.2


In [12]:
all_T = [1000, 2000, 3000, 5000]

table_vals = np.zeros((5, 6))
table_cols = ['T[сек]', 'p_rej', 'Q', 'p_free1', 'p_free2', 'k']
table_rows = [1, 2, 3, 4, 'Theor']


for i in range(len(all_T)):
    T = all_T[i]
    T_req = generate_request_time_exp(lmbd, T)
    T_processing = generate_process_time_exp(mu, len(T_req))

    processed_requests, fot_times, ffss = simulate(T_req, T_processing, T)

    p_rej = 1 - (processed_requests / len(T_req))
    q = processed_requests / len(T_req)
    p_free1 = 1 - (ffss[0] / np.sum(fot_times))
    p_free2 = 1 - (ffss[2] / np.sum(fot_times))
    k = 2 - (p_free1 + p_free2)

    table_vals[i, :] = [T, p_rej, q, p_free1, p_free2, k]

test_p = lmbd / mu

p0 = 1 / (1 + test_p + test_p**2 / 2)
p1 = test_p * p0
p2 = test_p**2 / 2 * p0

Q = 1 - p2
A = lmbd * Q
k = A / mu

table_vals[-1, :] = [0, p2, Q, p1, p1, k]
df = pd.DataFrame(table_vals, columns=table_cols, index=table_rows)


df

,T[сек],p_rej,Q,p_free1,p_free2,k
1,1000.0,0.383178,0.616822,0.309987,0.519184,1.170828
2,2000.0,0.437018,0.562982,0.336844,0.427317,1.235839
3,3000.0,0.403315,0.596685,0.338595,0.526566,1.134838
4,5000.0,0.397129,0.602871,0.344151,0.479102,1.176747
Theor,0.0,0.400000,0.600000,0.400000,0.400000,1.200000
